# Green data and sustainable ML
Explore how to reduce environmental impact of data and ML pipelines.

## How to read this notebook
Most code cells here are **estimators/simulations** (not precise measurements). They are designed to help you reason about key drivers of emissions:
- **Compute time** (how long something runs)
- **Power draw** (Watts)
- **Grid carbon intensity** (gCO2e per kWh)
- **Data center overhead** (PUE)
- **Data volume** (storage + transfer)

Each snippet prints a small "report" so you can immediately see what changes when you tweak parameters.


In [ ]:
# Imports
import time
from functools import lru_cache
from pprint import pprint
from typing import TypedDict

## Learning goals
- Map the data lifecycle to energy drivers (storage, transfer, compute).
- Estimate carbon intensity of workloads and spot hotspots.
- Apply efficiency patterns: pruning data, batching, caching, lower precision, and scheduling.
- Compare cloud regions by carbon intensity when choosing deployment.

### Why this matters
Machine learning models and data processing pipelines consume significant amounts of electricity. This energy use translates into carbon emissions depending on the **carbon intensity** of the electricity grid (which varies by region and time). By optimizing code and infrastructure choices, you can often reduce environmental footprint without sacrificing performance. This notebook focuses on practical, code-level changes you can make today.

## Quick reference & vocabulary
- **PUE (Power Usage Effectiveness):** total facility energy / IT equipment energy. Ideal is 1.0.
- **CI (Carbon Intensity):** gCO2e per kWh of electricity. Varies by region and time.
- **SCI (Software Carbon Intensity):** rate of carbon emissions per functional unit $R$. $SCI = ((E \cdot I) + M) / R$.
- **Embodied carbon:** CO2e emitted during hardware manufacturing and disposal.
- **Operational carbon:** CO2e emitted from running the software (electricity).
- **Demand shifting:** moving workloads to times (temporal) or regions (spatial) with lower CI.

## Main Green AI concepts (with sources)
- **Green Software Foundation Principles:** Energy efficiency, Hardware efficiency, Carbon awareness. [GSF](https://greensoftware.foundation/)
- **Energy and Policy Considerations (Strubell et al.):** Certain NLP training and architecture-search workloads can carry substantial energy and carbon costs; the paper argues for reporting and comparing those costs instead of treating compute as invisible. [Paper](https://arxiv.org/abs/1906.02243)
- **Green AI (Schwartz et al.):** Efficiency should be reported alongside model quality so progress does not depend only on increasing compute. [Paper](https://arxiv.org/abs/1907.10597)

Treat historical estimates as evidence about scale and measurement practice, not as universal emissions values for every modern model or data center.

![Typical evaporative cooling system in a data center](assets/data-center-cooling-system.jpg)

*Data-center energy use is not limited to servers: cooling and other facility systems are part of the overhead captured by PUE.*

Source: [U.S. Department of Energy, Cooling Water Efficiency Opportunities for Federal Data Centers](https://www.energy.gov/cmei/femp/cooling-water-efficiency-opportunities-federal-data-centers).

## Green patterns in code
- **Carbon awareness:** run jobs when the grid is greener (or choose a lower-CI region when allowed).
- **Energy efficiency:** optimize code to run faster or on less hardware.
- **Hardware efficiency:** use higher utilization or specialized hardware (GPU/TPU/NPU) when appropriate.
- **Data efficiency:** train on less data (curation, deduplication).
The code snippets below show concrete examples of applying these patterns.

### How the carbon estimate works (units + formula)
The first estimator uses a simple chain of conversions:
- Power in **Watts (W)** × time in **hours (h)** → energy in **Watt-hours (Wh)**
- Divide by 1000 → **kWh**
- Multiply by grid **carbon intensity** (gCO2e/kWh) → **grams of CO2e**

In symbols:
$$
E_{kWh} = \frac{hours \cdot watts}{1000}, \qquad CO2e_{g} = E_{kWh} \cdot CI
$$
Assumptions: CPU-only, constant watts, constant CI, no PUE yet (we add that later).


In [ ]:
# Simple carbon estimate for a data job
def estimate_job_impact(
    job_name: str,
    runtime_hours: float,
    watts: float = 65,
    carbon_intensity: float = 300,
):
    kwh_used = runtime_hours * watts / 1000
    co2e_grams = kwh_used * carbon_intensity

    print(f"--- Impact Report: {job_name} ---")
    print(f"Duration: {runtime_hours} hours")
    print(f"Power: {watts} Watts")
    print(f"Grid Intensity: {carbon_intensity} gCO2e/kWh")
    print(f"Total Energy: {kwh_used:.2f} kWh")
    print(f"Total Emissions: {co2e_grams:.2f} gCO2e")
    print("--------------------------------")
    return co2e_grams


# Scenario: ETL job running for 12 hours on a standard server (80W) in a coal-heavy region (450g)
job_co2e = estimate_job_impact(
    "Daily ETL (Coal Region)", runtime_hours=12, watts=80, carbon_intensity=450
)

In [ ]:
# Carbon Awareness: Spatial shifting to greener regions
# Different regions have different energy mixes (coal, hydro, nuclear, etc.).
# We also use PUE (Power Usage Effectiveness) to fold in facility overhead.
class RegionInfo(TypedDict):
    ci: float
    name: str
    pue: float


regions: dict[str, RegionInfo] = {
    "us-east-1": {
        "ci": 480.0,
        "name": "N. Virginia (illustrative fossil-heavy mix)",
        "pue": 1.2,
    },
    "eu-north-1": {
        "ci": 15.0,
        "name": "Stockholm (illustrative hydro/wind mix)",
        "pue": 1.1,
    },
    "eu-west-3": {
        "ci": 55.0,
        "name": "Paris (illustrative low-carbon mix)",
        "pue": 1.15,
    },
    "ap-south-1": {
        "ci": 720.0,
        "name": "Mumbai (illustrative coal-heavy mix)",
        "pue": 1.25,
    },
}


def pick_greenest_region(options: list[str]):
    print(f"Evaluating regions: {', '.join(options)}")
    best = min(options, key=lambda r: regions[r]["ci"] * regions[r]["pue"])
    worst = max(options, key=lambda r: regions[r]["ci"] * regions[r]["pue"])

    best_score = regions[best]["ci"] * regions[best]["pue"]
    worst_score = regions[worst]["ci"] * regions[worst]["pue"]
    pct_savings = (1 - best_score / worst_score) * 100

    print()
    print(f"[BEST CHOICE] {regions[best]['name']}")
    print(f"   Intensity: {regions[best]['ci']} gCO2e/kWh")
    print(f"   PUE: {regions[best]['pue']}")
    print(f"   Carbon-aware score (CI x PUE): {best_score:.1f}")
    print()
    print(f"[AVOID] {regions[worst]['name']} ({worst_score:.1f} combined score)")
    print(f"[POTENTIAL SAVINGS] {pct_savings:.1f}% reduction in operational carbon.")


pick_greenest_region(["us-east-1", "eu-west-3", "eu-north-1", "ap-south-1"])

**What this code does (region picker):** Chooses the region with the lowest approximate operational-carbon score from a list of allowed regions.

Step-by-step:
1. `regions` is a lookup table: each region has an illustrative **carbon intensity** (`ci`) and **PUE** (`pue`).
2. `pick_greenest_region(options)` compares a simple combined score: `ci * pue`.
3. It prints the best and worst options plus a rough percentage savings.

Limitations / real-world caveats:
- These values are rounded teaching placeholders dated January 2026; live carbon-aware scheduling needs current data.
- You may be constrained by data residency, latency, security, customer contracts, service availability, and cost.
- Some workloads cannot move regions at all, so temporal shifting or code efficiency may be the safer optimization.

### Challenge: Lower the estimate
Try to lower the estimate: reduce CPU hours (optimize queries), lower watts (right-size instances), or pick a region with lower carbon intensity.

### Solution
A reasonable improvement is to cut runtime, right-size the machine, and move to a lower-carbon grid if constraints allow. The next cell shows one simple before/after comparison.

In [ ]:
# Solution: reduce duration, right-size watts, and pick a lower-carbon region.
optimized_job_co2e = estimate_job_impact(
    "Daily ETL (Optimized)", runtime_hours=8, watts=60, carbon_intensity=160
)
reduction_pct = (1 - optimized_job_co2e / job_co2e) * 100
print(f"Reduction vs baseline: {reduction_pct:.1f}%")

### Why "deduplicate + prune columns" helps
This next code cell demonstrates **data efficiency**: reduce the size of your dataset before you store it, move it, or train on it.

What to look for in the output:
- `Deduplicated count`: fewer rows -> less storage + less downstream compute.
- `Kept columns`: fewer columns -> smaller records -> less transfer and memory usage.
- `[ROW REDUCTION]`: a simple percentage reduction indicator.

Note: this example deduplicates by a stable internal subject key so the sustainability example does not require raw personal identifiers.

In [ ]:
# Deduplicate and prune columns to reduce storage and transfer
# Expanded dataset simulating raw logs with an internal subject key.
records = [
    {
        "id": 1,
        "subject_key": "subj-001",
        "country": "FR",
        "timestamp": "2023-10-01T10:00:00",
        "browser": "Chrome",
        "os": "Mac",
    },
    {
        "id": 2,
        "subject_key": "subj-001",
        "country": "FR",
        "timestamp": "2023-10-01T10:05:00",
        "browser": "Chrome",
        "os": "Mac",
    },
    {
        "id": 3,
        "subject_key": "subj-002",
        "country": "DE",
        "timestamp": "2023-10-01T10:10:00",
        "browser": "Firefox",
        "os": "Linux",
    },
    {
        "id": 4,
        "subject_key": "subj-003",
        "country": "US",
        "timestamp": "2023-10-01T10:15:00",
        "browser": "Safari",
        "os": "iOS",
    },
    {
        "id": 5,
        "subject_key": "subj-002",
        "country": "DE",
        "timestamp": "2023-10-01T10:20:00",
        "browser": "Firefox",
        "os": "Linux",
    },
]


def optimize_dataset(rows, key, keep_columns):
    initial_count = len(rows)
    seen = set()
    result = []

    print(f"Processing {initial_count} records...")

    for row in rows:
        k = row[key]
        if k in seen:
            continue
        seen.add(k)
        pruned_row = {col: row[col] for col in keep_columns if col in row}
        result.append(pruned_row)

    final_count = len(result)
    reduction = 100 * (1 - final_count / initial_count)

    print(f"Deduplicated count: {final_count}")
    print(f"Kept columns: {keep_columns}")
    print(f"[ROW REDUCTION] {reduction:.1f}%")
    return result


optimized = optimize_dataset(
    records, key="subject_key", keep_columns=["id", "subject_key", "country"]
)
pprint(optimized)

## Mini-case: Audit this pipeline
A daily ETL job runs in `us-east-1` at 10 AM, processes 1 TB of raw logs, keeps every source column, rewrites the full output on every run, and stores all historical results in hot storage indefinitely. The product serves EU and US customers, has a four-hour freshness target, and the team has not documented whether workloads or personal data may move to another region.

Identify the main footprint drivers, propose changes in priority order, list the constraints and evidence you must check, and choose a first action. Do not invent a precise savings percentage when the required measurements are missing. Complete the structured cell below before reading **Solution 1**.

In [ ]:
# Capture the main footprint drivers and your recommended changes for the mini-case.
pipeline_audit = {
    "hotspots": [],
    "recommended_changes": [],
    "constraints_to_check": [],
    "evidence_needed": [],
    "first_action": "",
    "expected_impact": "",
    "reasoning": "",
}
pipeline_audit

### Quantization / lower precision (why it matters)
Model parameters take memory. Memory affects:
- how many GPUs/CPUs you need,
- how much energy you burn moving weights around, and
- whether you can serve efficiently.

The next cell is a **back-of-the-envelope** calculation: parameters × bytes-per-parameter → total bytes → GB. It then reports % savings relative to float32.


In [ ]:
# Hardware Efficiency: Quantization / Lower Precision
# Simulating model weights size reduction for a large language model (e.g., 7B parameters)
model_params = 7_000_000_000


def estimate_model_footprint(params: int):
    precisions = {
        "float32 (Full)": 4,
        "float16 (Half)": 2,
        "bfloat16 (Brain float)": 2,
        "int8 (Quantized)": 1,
        "int4 (Aggressive)": 0.5,
    }

    print(f"--- Memory Footprint for {params / 1e9:.1f}B Parameter Model ---")
    base_size = params * precisions["float32 (Full)"] / (1024**3)  # GB

    for name, bytes_per_param in precisions.items():
        size_gb = (params * bytes_per_param) / (1024**3)
        savings = 100 * (1 - size_gb / base_size)
        print(f"{name:<20} | {size_gb:>6.2f} GB | Savings: {savings:>5.1f}%")


estimate_model_footprint(model_params)

**What this code does:** Estimates how much memory a large model would need under different numeric precisions (float32/float16/int8/int4).
- It assumes a fixed number of parameters (here: 7B).
- It assigns a simplified “bytes per parameter” for each precision.
- It prints the resulting GB and the % savings versus float32.

How to interpret it: this is a proxy for potential **hardware and energy** savings, not an energy measurement. The estimate covers model weights only; serving also needs memory and compute for activations, the KV cache, runtime overhead, and sometimes quantization metadata. Benchmark latency, throughput, quality, and measured energy on the target hardware before claiming a sustainability gain.


## Exercises
1. Complete the pipeline-audit mini-case. Prioritize changes that reduce unnecessary work before considering region or schedule shifts, and distinguish measured evidence from assumptions.
2. Add caching to a data transform you commonly run and measure runtime savings.
3. Draft a lifecycle policy for cold or archival storage and identify the provider data needed before estimating yearly energy savings.
4. Identify legal, contractual, technical, and product constraints that may prevent region-hopping for greener grids.

## Solutions

The following cells contain example solutions for the exercises above.

### Solution 1: Pipeline audit

This example starts with changes that remove waste in any region. It treats relocation as a later option because legal, contractual, latency, and service constraints may block it, and it does not claim unsupported numerical savings.

In [ ]:
# Solution exercise 1: evidence-based pipeline audit.
pipeline_audit_solution = {
    "hotspots": [
        "Full rewrites repeat compute and data transfer on every run",
        "Unused columns increase processing, transfer, and storage",
        "Indefinite hot storage keeps unnecessary historical outputs",
        "Fixed timing and region may miss lower-carbon operating windows",
    ],
    "recommended_changes": [
        "Measure runtime, power or provider energy data, bytes read and written, and storage growth",
        "Use incremental processing instead of rewriting the complete dataset",
        "Prune unused columns and deduplicate before writing outputs",
        "Define retention and lifecycle rules based on actual business and legal needs",
        "Evaluate temporal or regional shifting only after checking constraints",
    ],
    "constraints_to_check": [
        "data residency and international-transfer requirements",
        "customer and supplier contracts",
        "four-hour freshness target and user latency",
        "service, hardware, and lower-carbon scheduling availability",
    ],
    "evidence_needed": [
        "current runtime and workload utilization",
        "current grid intensity and data-center overhead",
        "column usage and incremental-change rate",
        "storage volume, access frequency, and retention obligations",
    ],
    "first_action": (
        "Instrument the workload, then stop full rewrites and unused-column processing"
    ),
    "expected_impact": (
        "Likely reduction across compute, transfer, and storage; quantify after measurement"
    ),
    "reasoning": (
        "Removing unnecessary work is actionable without moving customer data and creates a "
        "measured baseline for evaluating later scheduling or region changes."
    ),
}
pprint(pipeline_audit_solution)

### Solution 2: Caching to reduce redundant compute
We use Python's built-in `lru_cache` to avoid re-running the same expensive transformation for the same inputs.

What to look for:
- The **first run** includes cache misses → slower.
- The **second run** reuses cached results → faster.

Why it can help: avoiding repeated deterministic computation can reduce runtime and energy use, especially in batch ETL and feature pipelines. A cache also consumes memory or storage and needs invalidation, so compare the resources it saves with the resources it adds.


In [ ]:
# Simulate an expensive data transformation
@lru_cache(maxsize=128)
def expensive_transform(data_id):
    time.sleep(0.5)  # Simulate 500ms of heavy compute
    return f"Transformed_{data_id}"


def run_pipeline(ids):
    start = time.time()
    for i in ids:
        _ = expensive_transform(i)
    end = time.time()
    print(f"Processed {len(ids)} items in {end - start:.2f} seconds")


data_ids = [1, 2, 1, 3, 2, 4, 1, 5]  # Note duplicates: 1 and 2 appear multiple times

print("First run (some cache misses):")
run_pipeline(data_ids)

print("\nSecond run (all cache hits):")
run_pipeline(data_ids)

### Solution 3: Lifecycle Policy Draft

**Goal:** Stop keeping rarely accessed data in an unnecessarily expensive and resource-intensive service tier.

**Policy:**
1.  **Frequent-access tier:** data less than 30 days old, when low-latency retrieval is an evidenced requirement.
2.  **Infrequent-access tier:** data from 30 to 90 days old when slower or less frequent retrieval is acceptable.
3.  **Archive tier:** data older than 90 days only when a documented need still justifies retention; record retrieval delay and restoration cost.
4.  **Delete:** Logs > 1 year old (unless required for compliance or active investigations).

**Estimated Savings:**
Do not infer a precise energy saving from a storage-class label alone. Ask the provider for service-level energy or carbon data where available, measure stored volume and access frequency, and report cost or capacity reductions separately from verified emissions reductions.

---

### Solution 4: Constraints on Region Hopping

A greener region is not automatically a permissible region. Check constraints such as:

1.  **Data residency and transfer rules:** personal data may need to stay in particular jurisdictions, and cross-border moves may require a transfer mechanism plus a risk assessment.
2.  **Customer or sector requirements:** contracts, procurement terms, healthcare, finance, or public-sector rules may pin workloads to specific regions.
3.  **Latency and user experience:** moving a latency-sensitive system too far away can create product or safety problems.
4.  **Service availability and cost:** the greenest region may not offer the required GPU type, managed service, or budget profile.